# Customer Support Agent — Braintrust Evaluation

This notebook evaluates the LangGraph customer-support classifier using **Braintrust**.

The structure now follows the evaluation lifecycle used in the course:

## Flow

```text
Generate tickets
      ↓
Golden dataset
      ↓
Classifier V1
      ↓
ExactMatch baseline
      ↓
Failure analysis
      ↓
Human review subset
      ↓
Prompt V2
      ↓
ExactMatch + V1/V2 comparison
      ↓
Failure → regression dataset
      ↓
Regression experiment
      ↓
LLM judges
      ↓
Judge vs human calibration
      ↓
Diagnose disagreements
```

## 1. Install dependencies

## 2. Environment and API keys

This repository uses a local `.env` file for secrets.

1. Copy `.env.example` to `.env`.
2. Add your own `OPENAI_API_KEY` and `BRAINTRUST_API_KEY`.
3. Never commit `.env` to Git.

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

required_keys = ["OPENAI_API_KEY", "BRAINTRUST_API_KEY"]
missing = [key for key in required_keys if not os.getenv(key)]

if missing:
    raise EnvironmentError(
        f"Missing environment variables: {', '.join(missing)}. "
        "Create a .env file from .env.example and add your API keys."
    )

print("Environment configured.")

Environment configured.


## 3. Braintrust tracing

Using Braintrust's LangChain/LangGraph callback integration. The classifier code itself does not need Braintrust-specific changes.

In [3]:
import braintrust
from braintrust.integrations.langchain import BraintrustCallbackHandler, set_global_handler

PROJECT_NAME = "customer-support-braintrust-evals"

braintrust.init_logger(project=PROJECT_NAME)
set_global_handler(BraintrustCallbackHandler())

print("Braintrust project:", PROJECT_NAME)

Braintrust project: customer-support-braintrust-evals


## 4. Ticket categories and dataset generation

In [4]:
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from typing import List
from collections import Counter

CATEGORIES = [
    "order_status", "refund_request", "product_issue", "account_help", "other"
]

SEED_TICKETS = [
    ("Hi, I ordered a blender 5 days ago and the tracking page hasn't updated. Can you tell me where it is?", "order_status"),
    ("I never received my order and I want my money back.", "order_status"),
    ("Order shows delivered last Tuesday but it's not at my door, my neighbor's, or the mailroom. What now?", "order_status"),
    ("The website said 2-day shipping. It's been 9 days. Are you kidding me?", "order_status"),
    ("Tracking link in the email just spins forever. Order #88421.", "order_status"),
    ("I'd like to return the headphones I bought last week and get my money back. They're unopened.", "refund_request"),
    ("Where is my refund? I returned the item two weeks ago and still nothing on my card.", "refund_request"),
    ("Please cancel order 99021 and refund my card. I no longer need it.", "refund_request"),
    ("I was charged $89 but the website showed $79 at checkout. Please refund the difference.", "refund_request"),
    ("Returning these for store credit is fine but honestly I'd prefer cash back to my original card.", "refund_request"),
    ("The coffee maker arrived with a cracked carafe. Really disappointed.", "product_issue"),
    ("My laptop arrived damaged and I want a full refund, not a replacement.", "product_issue"),
    ("You sent me a size medium shirt but I ordered a large. Second time this has happened.", "product_issue"),
    ("The product description said 'wireless' but I had to buy a separate dongle to use it. Misleading.", "product_issue"),
    ("Item itself works fine but the box was crushed and the instruction manual is missing.", "product_issue"),
    ("I can't log into my account — it keeps saying my password is wrong even after I reset it.", "account_help"),
    ("How do I update the credit card on file? I don't see the option anywhere in settings.", "account_help"),
    ("The website won't let me check out — it keeps logging me out mid-payment.", "account_help"),
    ("Please remove my old shipping address. I moved last month and don't want stuff going there.", "account_help"),
    ("I keep getting 2FA codes I didn't request. Is someone trying to access my account?", "account_help"),
    ("Do you guys ship to Canada? Couldn't find it on the FAQ page.", "other"),
    ("Just wanted to say the customer service rep I spoke to yesterday was amazing. Thank you!", "other"),
    ("Is the red version of SKU-1140 back in stock?", "other"),
    ("Do you offer a student discount? Couldn't find one at checkout.", "other"),
    ("When's your next sale? My birthday is coming up and I'd love to splurge.", "other"),
]

EXPANSIONS_PER_SEED = 3
generator_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.9)

class Paraphrases(BaseModel):
    variations: List[str] = Field(description="Realistic paraphrases of the original ticket")

def expand(seed_text, n):
    structured = generator_llm.with_structured_output(Paraphrases)
    out = structured.invoke(
        f"""You write realistic e-commerce customer-support tickets.
Produce {n} short paraphrases of the ticket below. Keep the same underlying intent,
but vary tone and phrasing. Preserve ambiguity if the original has it.

Original ticket:
{seed_text}"""
    )
    return out.variations

tickets = []
for text, label in SEED_TICKETS:
    tickets.append({"id": f"t{len(tickets):03d}", "text": text,
                    "true_category": label, "source": "seed"})
    for variation in expand(text, EXPANSIONS_PER_SEED):
        tickets.append({"id": f"t{len(tickets):03d}", "text": variation,
                        "true_category": label, "source": "synthetic"})

print(f"Generated {len(tickets)} tickets.")
print(Counter(t["true_category"] for t in tickets))

Generated 100 tickets.
Counter({'order_status': 20, 'refund_request': 20, 'product_issue': 20, 'account_help': 20, 'other': 20})


## 5. LangGraph classifier

In [5]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langchain_core.prompts import ChatPromptTemplate

CLASSIFIER_PROMPT = """You are a triage system for an e-commerce support inbox.

Classify the customer's ticket into EXACTLY ONE of these categories:

- order_status: questions about where an order is, tracking, delivery ETA
- refund_request: the customer wants their money back
- product_issue: the item arrived broken, wrong, defective, or not as described
- account_help: login, password, address, payment method changes
- other: anything that doesn't fit the above (general questions, feedback, browsing)

Return only the category key.

Ticket:
{ticket_text}
"""

class Classification(BaseModel):
    category: Literal["order_status", "refund_request", "product_issue", "account_help", "other"]
    reasoning: str = Field(description="One short sentence explaining the choice.")

class AgentState(TypedDict):
    ticket_text: str
    category: str
    reasoning: str

def build_agent(prompt_template):
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).with_structured_output(Classification)
    prompt = ChatPromptTemplate.from_template(prompt_template)

    def classify_node(state):
        result = (prompt | llm).invoke({"ticket_text": state["ticket_text"]})
        return {
            "ticket_text": state["ticket_text"],
            "category": result.category,
            "reasoning": result.reasoning,
        }

    graph = StateGraph(AgentState)
    graph.add_node("classify", classify_node)
    graph.add_edge(START, "classify")
    graph.add_edge("classify", END)
    return graph.compile()

agent = build_agent(CLASSIFIER_PROMPT)

sample = agent.invoke({"ticket_text": tickets[0]["text"], "category": "", "reasoning": ""})
print("Ticket:", tickets[0]["text"])
print("Prediction:", sample["category"])
print("Reasoning:", sample["reasoning"])

Ticket: Hi, I ordered a blender 5 days ago and the tracking page hasn't updated. Can you tell me where it is?
Prediction: order_status
Reasoning: The customer is inquiring about the status and tracking of their order.


### Trace checkpoint

Open Braintrust and verify the smoke test is visible.

## 6. Create the Braintrust golden dataset

In [6]:
from braintrust import init_dataset

DATASET_NAME = "customer-support-golden-100"

bt_dataset = init_dataset(
    project=PROJECT_NAME,
    name=DATASET_NAME,
    description="Golden dataset for customer-support ticket classification",
)

for t in tickets:
    bt_dataset.insert(
        input=t["text"],
        expected=t["true_category"],
        metadata={"ticket_id": t["id"], "source": t["source"]},
        id=t["id"],
    )

bt_dataset.flush()
print("Dataset ready:", DATASET_NAME)

Dataset ready: customer-support-golden-100


Braintrust datasets are versioned objects and can be reused across experiments. A useful Braintrust workflow is to promote interesting failures/traces into datasets so they become future regression cases.

## 7. Baseline scorer — ExactMatch

For the baseline we use `ExactMatch`.

It answers one deterministic question:

> Does the predicted category exactly match the golden category?


In [7]:
from autoevals import ExactMatch

print("Baseline scorer ready: ExactMatch")

Baseline scorer ready: ExactMatch


## 8. Define the task

In [8]:
def classify_for_eval(input,*args, **kwargs):
    out = agent.invoke({
        "ticket_text": input,
        "category": "",
        "reasoning": "",
    })
    return out["category"]

## 9. Experiment V1 — baseline

V1 is evaluated against the full golden dataset using **ExactMatch** scorer.

This gives us a clean, interpretable baseline before adding human or LLM-based evaluation.

In [9]:
from braintrust import EvalAsync

baseline = await EvalAsync(
    PROJECT_NAME,
    experiment_name="classifier-v1-baseline",
    data=bt_dataset,
    task=classify_for_eval,
    scores=[ExactMatch],
    metadata={
        "task_model": "gpt-4o-mini",
        "prompt_version": "v1-baseline",
        "dataset": DATASET_NAME,
        "evaluation_stage": "baseline",
    },
)

print(baseline)

Experiment classifier-v1-baseline is running at https://www.braintrust.dev/app/The%20Gen%20Academy/p/customer-support-braintrust-evals/experiments/classifier-v1-baseline
customer-support-braintrust-evals [experiment_name=classifier-v1-baseline] (data): 100it [00:00, 22942.26it/s]


customer-support-braintrust-evals [experiment_name=classifier-v1-baseline] (tasks):   0%|          | 0/100 [00…


=========================SUMMARY=========================
92.00% 'ExactMatch' score

1 llm_calls
0 tool_calls
0 errors
0 llm_errors
0 tool_errors
214.13tok prompt_tokens
0tok prompt_cached_tokens
0tok prompt_cache_creation_tokens
0tok prompt_cache_creation_5m_tokens
0tok prompt_cache_creation_1h_tokens
24.02tok completion_tokens
0tok completion_reasoning_tokens
238.15tok total_tokens
0.00$ estimated_cost
5.33s duration
1.15s llm_duration

See results for classifier-v1-baseline at https://www.braintrust.dev/app/The%20Gen%20Academy/p/customer-support-braintrust-evals/experiments/classifier-v1-baseline
EvalResultWithSummary(summary=ExperimentSummary(project_name='customer-support-braintrust-evals', project_id='19a7b66c-64ed-44ec-8291-53a35ba6e9a0', experiment_id='a95594c4-0a61-4fd2-9eca-7c9ae2764109', experiment_name='classifier-v1-baseline', project_url='https://www.braintrust.dev/app/The%20Gen%20Academy/p/customer-support-braintrust-evals', experiment_url='https://www.braintrust.dev/ap

Open the V1 experiment in Braintrust and inspect the ExactMatch score, failed rows, traces, latency/token metrics, and errors.

## 10. Local metrics + failure analysis

The aggregate score tells us how often the classifier is correct.

The confusion matrix and failed rows tell us **where it fails**, which is what we use to design the next iteration.

In [10]:
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

def run_predictions(agent_to_use, tickets):
    rows = []
    for t in tqdm(tickets, desc="Classifying"):
        out = agent_to_use.invoke({
            "ticket_text": t["text"],
            "category": "",
            "reasoning": "",
        })
        rows.append({
            "id": t["id"],
            "ticket_text": t["text"],
            "true_category": t["true_category"],
            "predicted_category": out["category"],
            "reasoning": out["reasoning"],
            "correct": t["true_category"] == out["category"],
        })
    return pd.DataFrame(rows)

def evaluate_df(df, label):
    y_true, y_pred = df["true_category"], df["predicted_category"]
    acc = accuracy_score(y_true, y_pred)
    print(f"=== {label} ===")
    print(f"Accuracy: {acc:.2%}")
    print(classification_report(y_true, y_pred, labels=CATEGORIES, zero_division=0))
    display(pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=CATEGORIES),
        index=CATEGORIES, columns=CATEGORIES
    ))
    return acc

results_v1 = run_predictions(agent, tickets)
acc_v1 = evaluate_df(results_v1, "V1 baseline")

results_v1["validator_comment"] = ""
results_v1["failure_category"] = ""
results_v1.to_csv("braintrust_results_v1.csv", index=False)

display(results_v1[~results_v1["correct"]].head(20))

Classifying:   0%|          | 0/100 [00:00<?, ?it/s]Retrying request after error: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))
Sleeping for 0.5 seconds
Classifying: 100%|██████████| 100/100 [01:38<00:00,  1.01it/s]

=== V1 baseline ===
Accuracy: 92.00%
                precision    recall  f1-score   support

  order_status       1.00      0.80      0.89        20
refund_request       0.71      1.00      0.83        20
 product_issue       1.00      0.80      0.89        20
  account_help       1.00      1.00      1.00        20
         other       1.00      1.00      1.00        20

      accuracy                           0.92       100
     macro avg       0.94      0.92      0.92       100
  weighted avg       0.94      0.92      0.92       100



,order_status,refund_request,product_issue,account_help,other
order_status,16,4,0,0,0
refund_request,0,20,0,0,0
product_issue,0,4,16,0,0
account_help,0,0,0,20,0
other,0,0,0,0,20


,id,ticket_text,true_category,predicted_category,reasoning,correct,validator_comment,failure_category
4,t004,I never received my order and I want my money ...,order_status,refund_request,The customer is requesting a refund because th...,False,,
5,t005,"I didn't get my order, and I'm requesting a re...",order_status,refund_request,The customer is requesting a refund due to not...,False,,
6,t006,"My order hasn't arrived, so I'd like to initia...",order_status,refund_request,The customer wants to initiate a return for th...,False,,
7,t007,I haven't received my package and I would like...,order_status,refund_request,The customer is requesting a refund due to not...,False,,
44,t044,My laptop arrived damaged and I want a full re...,product_issue,refund_request,The customer is requesting a full refund for a...,False,,
45,t045,I received a damaged laptop and would like to ...,product_issue,refund_request,The customer is requesting a full refund for a...,False,,
46,t046,The laptop I ordered arrived in poor condition...,product_issue,refund_request,The customer wants their money back for a prod...,False,,
47,t047,"Unfortunately, my laptop came damaged, and I a...",product_issue,refund_request,The customer is requesting a complete refund f...,False,,


## 11. Human review in Braintrust UI

For human annotation, we use **Braintrust Human Review**.

We will create a small review set containing:

- all V1 failures
- 12 randomly sampled correct predictions

The review set contains both difficult and normal examples, which makes it useful later for calibrating an automated LLM judge.

### Human-review rubric

In Braintrust, configure the project-level review rubric under:

**Project → Settings → Human review**

Recommended fields for this project:

- **HumanCorrectness** — pass/fail  
  `Correct = 1`, `Incorrect = 0`
- **HumanExpectedCategory** — categorical  
  `order_status`, `refund_request`, `product_issue`, `account_help`, `other`
- **ReviewerNotes** — free-form comment

For judge calibration, `HumanCorrectness` is the most important field because the LLM judge will also return a binary correctness score.

In [12]:
# Build the fixed review subset once.
failures_for_review = results_v1[~results_v1["correct"]].copy()

correct_for_review = (
    results_v1[results_v1["correct"]]
    .sample(n=min(12, int(results_v1["correct"].sum())), random_state=42)
    .copy()
)

review_cases = (
    pd.concat([failures_for_review, correct_for_review], ignore_index=True)
    .drop_duplicates(subset=["id"])
)

print(f"Selected {len(review_cases)} examples for Braintrust Human Review.")
display(
    review_cases[
        ["id", "ticket_text", "true_category", "predicted_category", "reasoning"]
    ]
)

Selected 20 examples for Braintrust Human Review.


,id,ticket_text,true_category,predicted_category,reasoning
0,t004,I never received my order and I want my money ...,order_status,refund_request,The customer is requesting a refund because th...
1,t005,"I didn't get my order, and I'm requesting a re...",order_status,refund_request,The customer is requesting a refund due to not...
2,t006,"My order hasn't arrived, so I'd like to initia...",order_status,refund_request,The customer wants to initiate a return for th...
3,t007,I haven't received my package and I would like...,order_status,refund_request,The customer is requesting a refund due to not...
4,t044,My laptop arrived damaged and I want a full re...,product_issue,refund_request,The customer is requesting a full refund for a...
5,t045,I received a damaged laptop and would like to ...,product_issue,refund_request,The customer is requesting a full refund for a...
6,t046,The laptop I ordered arrived in poor condition...,product_issue,refund_request,The customer wants their money back for a prod...
7,t047,"Unfortunately, my laptop came damaged, and I a...",product_issue,refund_request,The customer is requesting a complete refund f...
8,t048,You sent me a size medium shirt but I ordered ...,product_issue,product_issue,"The customer received the wrong size shirt, wh..."
9,t026,Could you please check on my refund? I returne...,refund_request,refund_request,The customer is inquiring about the status of ...


### Create a dedicated Braintrust dataset for review

The selected examples are stored as a small dataset so the same cases can be reused for:

1. human review
2. LLM-judge calibration
3. later scorer comparisons

The human annotations themselves are entered in the **Braintrust UI**, not in Python.

In [13]:
HUMAN_REVIEW_DATASET_NAME = "customer-support-human-review-set"

human_review_dataset = init_dataset(
    project=PROJECT_NAME,
    name=HUMAN_REVIEW_DATASET_NAME,
    description="V1 failures plus sampled correct cases for human review and judge calibration",
)

for _, row in review_cases.iterrows():
    human_review_dataset.insert(
        input=row["ticket_text"],
        expected=row["true_category"],
        metadata={
            "ticket_id": row["id"],
            "v1_prediction": row["predicted_category"],
            "v1_reasoning": row["reasoning"],
            "selected_for": "human-review",
        },
        id=f"review-{row['id']}",
    )

human_review_dataset.flush()

print("Human-review dataset ready:", HUMAN_REVIEW_DATASET_NAME)

Human-review dataset ready: customer-support-human-review-set


### Run the review-set experiment

This experiment gives the reviewer the exact V1 outputs that need to be inspected.

After this cell finishes:

1. Open the experiment link in Braintrust.
2. Go to the project's **Review / Human review** workflow.
3. Assign or select the rows from `human-review-v1`.
4. For each row, record:
   - `HumanCorrectness`
   - `HumanExpectedCategory`
   - `ReviewerNotes`
5. Mark the review complete.

You only need to do this **once for this calibration set**. Future classifier runs reuse the same reviewed examples unless the task, taxonomy, or evaluation standard changes.

In [14]:
def classify_review_case(input, *args, **kwargs):
    out = agent.invoke({
        "ticket_text": input,
        "category": "",
        "reasoning": "",
    })
    return out["category"]

human_review_experiment = await EvalAsync(
    PROJECT_NAME,
    experiment_name="human-review-v1",
    data=human_review_dataset,
    task=classify_review_case,
    scores=[ExactMatch],
    metadata={
        "task_model": "gpt-4o-mini",
        "prompt_version": "v1-baseline",
        "purpose": "human-review-calibration-set",
    },
)

print(human_review_experiment)

Experiment human-review-v1 is running at https://www.braintrust.dev/app/The%20Gen%20Academy/p/customer-support-braintrust-evals/experiments/human-review-v1
customer-support-braintrust-evals [experiment_name=human-review-v1] (data): 20it [00:00, 49113.63it/s]


customer-support-braintrust-evals [experiment_name=human-review-v1] (tasks):   0%|          | 0/20 [00:00<?, ?…


=========================SUMMARY=========================
human-review-v1 compared to classifier-v1-baseline:
60.00% (-) 'ExactMatch' score	(0 improvements, 0 regressions)

1 (-) 'llm_calls'                      	(0 improvements, 0 regressions)
0 (-) 'tool_calls'                     	(0 improvements, 0 regressions)
0 (-) 'errors'                         	(0 improvements, 0 regressions)
0 (-) 'llm_errors'                     	(0 improvements, 0 regressions)
0 (-) 'tool_errors'                    	(0 improvements, 0 regressions)
212.20tok (-) 'prompt_tokens'                  	(0 improvements, 0 regressions)
0tok (-) 'prompt_cached_tokens'           	(0 improvements, 0 regressions)
0tok (-) 'prompt_cache_creation_tokens'   	(0 improvements, 0 regressions)
0tok (-) 'prompt_cache_creation_5m_tokens'	(0 improvements, 0 regressions)
0tok (-) 'prompt_cache_creation_1h_tokens'	(0 improvements, 0 regressions)
23.45tok (+05.00%) 'completion_tokens'              	(3 improvements, 2 regressions)
0

### Checkpoint: switch to Braintrust UI

Complete the human review in Braintrust UI.

> Human review establishes a trusted reference on a smaller set. We do not manually annotate every experiment run.

Braintrust keeps human scores alongside traces and experiment results, so the same reviewed cases can later be compared with automated scorer results.

Once the review is complete, return to the notebook and continue with V2.

## 12. Focused prompt change — V2

V1 failure analysis showed that the classifier could over-prioritize a requested remedy such as "refund" instead of the customer's underlying problem.

V2 adds explicit disambiguation rules so that the **primary issue** determines the category.

In [15]:
IMPROVED_PROMPT = """You are a triage system for an e-commerce support inbox.

Classify the customer's ticket into EXACTLY ONE of these categories:

- order_status: questions about where an order is, tracking, delivery ETA, or non-delivery
- refund_request: the customer is asking for their money back (and the item itself is fine, or already returned)
- product_issue: the item arrived broken, wrong, defective, or not as described — even if the customer also asks for a refund as the remedy
- account_help: login, password, address, or payment method changes
- other: general questions, feedback, browsing, anything not covered above

Disambiguation rules:
1. Damaged/wrong/defective item → product_issue regardless of requested remedy.
2. Never received the order → order_status even if they mention wanting money back.
3. Asking about a refund already initiated → refund_request.
4. Site bugs that prevent checkout → account_help.

First identify the customer's primary problem, then choose the category.

Ticket:
{ticket_text}
"""

agent_v2 = build_agent(IMPROVED_PROMPT)

## 13. Experiment V2 — same metric, same dataset

V2 is evaluated with the same golden dataset and the same `ExactMatch` scorer.

Keeping the metric fixed makes the V1 → V2 comparison easy to interpret.

In [16]:
def classify_v2_for_eval(input, *args, **kwargs):
    out = agent_v2.invoke({
        "ticket_text": input,
        "category": "",
        "reasoning": "",
    })
    return out["category"]

v2 = await EvalAsync(
    PROJECT_NAME,
    experiment_name="classifier-v2-focused-prompt",
    data=bt_dataset,
    task=classify_v2_for_eval,
    scores=[ExactMatch],
    metadata={
        "task_model": "gpt-4o-mini",
        "prompt_version": "v2-focused-disambiguation",
        "dataset": DATASET_NAME,
        "evaluation_stage": "iteration",
    },
    base_experiment_name="classifier-v1-baseline",
)

print(v2)

Experiment classifier-v2-focused-prompt is running at https://www.braintrust.dev/app/The%20Gen%20Academy/p/customer-support-braintrust-evals/experiments/classifier-v2-focused-prompt
customer-support-braintrust-evals [experiment_name=classifier-v2-focused-prompt] (data): 100it [00:00, 197564.96it/s]


customer-support-braintrust-evals [experiment_name=classifier-v2-focused-prompt] (tasks):   0%|          | 0/1…


=========================SUMMARY=========================
classifier-v2-focused-prompt compared to classifier-v1-baseline:
100.00% (+08.00%) 'ExactMatch' score	(8 improvements, 0 regressions)

1 (-) 'llm_calls'                      	(0 improvements, 0 regressions)
0 (-) 'tool_calls'                     	(0 improvements, 0 regressions)
0 (-) 'errors'                         	(0 improvements, 0 regressions)
0 (-) 'llm_errors'                     	(0 improvements, 0 regressions)
0 (-) 'tool_errors'                    	(0 improvements, 0 regressions)
313.13tok (+9900.00%) 'prompt_tokens'                  	(0 improvements, 100 regressions)
0tok (-) 'prompt_cached_tokens'           	(0 improvements, 0 regressions)
0tok (-) 'prompt_cache_creation_tokens'   	(0 improvements, 0 regressions)
0tok (-) 'prompt_cache_creation_5m_tokens'	(0 improvements, 0 regressions)
0tok (-) 'prompt_cache_creation_1h_tokens'	(0 improvements, 0 regressions)
26.30tok (+228.00%) 'completion_tokens'              	(1

**Important Braintrust comparison point:** V2 is explicitly linked to V1 as its base experiment. Braintrust can use that relationship to surface score deltas, improvements, and regressions.

## 14. Local V1 vs V2 regression check

We inspect not only the overall score, but also:

- wins: V1 wrong → V2 correct
- regressions: V1 correct → V2 wrong
- prediction flips between versions

In [17]:
results_v2 = run_predictions(agent_v2, tickets)
acc_v2 = evaluate_df(results_v2, "V2 focused prompt")

comparison = results_v1.merge(
    results_v2[["id", "predicted_category", "reasoning", "correct"]],
    on="id",
    suffixes=("_v1", "_v2"),
)

flipped = comparison[
    comparison["predicted_category_v1"] != comparison["predicted_category_v2"]
]
wins = flipped[(~flipped["correct_v1"]) & (flipped["correct_v2"])]
regressions = flipped[(flipped["correct_v1"]) & (~flipped["correct_v2"])]

print(f"Accuracy V1: {acc_v1:.2%}")
print(f"Accuracy V2: {acc_v2:.2%}")
print(f"Delta: {(acc_v2 - acc_v1):+.2%}")
print(f"Changed: {len(flipped)} | Wins: {len(wins)} | Regressions: {len(regressions)}")

display(flipped[
    ["id", "ticket_text", "true_category",
     "predicted_category_v1", "predicted_category_v2"]
])

Classifying: 100%|██████████| 100/100 [01:46<00:00,  1.07s/it]

=== V2 focused prompt ===
Accuracy: 100.00%
                precision    recall  f1-score   support

  order_status       1.00      1.00      1.00        20
refund_request       1.00      1.00      1.00        20
 product_issue       1.00      1.00      1.00        20
  account_help       1.00      1.00      1.00        20
         other       1.00      1.00      1.00        20

      accuracy                           1.00       100
     macro avg       1.00      1.00      1.00       100
  weighted avg       1.00      1.00      1.00       100



,order_status,refund_request,product_issue,account_help,other
order_status,20,0,0,0,0
refund_request,0,20,0,0,0
product_issue,0,0,20,0,0
account_help,0,0,0,20,0
other,0,0,0,0,20


Accuracy V1: 92.00%
Accuracy V2: 100.00%
Delta: +8.00%
Changed: 8 | Wins: 8 | Regressions: 0


,id,ticket_text,true_category,predicted_category_v1,predicted_category_v2
4,t004,I never received my order and I want my money ...,order_status,refund_request,order_status
5,t005,"I didn't get my order, and I'm requesting a re...",order_status,refund_request,order_status
6,t006,"My order hasn't arrived, so I'd like to initia...",order_status,refund_request,order_status
7,t007,I haven't received my package and I would like...,order_status,refund_request,order_status
44,t044,My laptop arrived damaged and I want a full re...,product_issue,refund_request,product_issue
45,t045,I received a damaged laptop and would like to ...,product_issue,refund_request,product_issue
46,t046,The laptop I ordered arrived in poor condition...,product_issue,refund_request,product_issue
47,t047,"Unfortunately, my laptop came damaged, and I a...",product_issue,refund_request,product_issue


## 15. Failure → regression dataset

The known V1 failures are promoted into a dedicated Braintrust dataset.

This converts a one-time failure analysis into a reusable regression suite.

In [18]:
important_failures = results_v1[results_v1["correct"] == False].head(10)

REGRESSION_DATASET_NAME = "customer-support-regression-cases"

regression_dataset = init_dataset(
    project=PROJECT_NAME,
    name=REGRESSION_DATASET_NAME,
    description="Important classifier failures promoted to regression tests",
)

for _, row in important_failures.iterrows():
    regression_dataset.insert(
        input=row["ticket_text"],
        expected=row["true_category"],
        metadata={
            "ticket_id": row["id"],
            "source": "baseline-failure",
        },
        id=f"reg-{row['id']}",
    )

regression_dataset.flush()
print(f"Promoted {len(important_failures)} failures to {REGRESSION_DATASET_NAME}.")

Promoted 8 failures to customer-support-regression-cases.


This is the workflow we specifically want to investigate:

```text
trace / failure
      ↓
keep as dataset case
      ↓
future experiment
      ↓
catch regression
```

Braintrust documents this production/trace → dataset → eval loop as a way to continuously expand regression coverage.

## 16. Run the regression suite

Run V2 against only the known V1 failure cases.

This verifies that the targeted failure mode stays fixed independently of the overall 100-case score.

In [19]:
regression_eval = await EvalAsync(
    PROJECT_NAME,
    experiment_name="classifier-v2-regression-suite",
    data=regression_dataset,
    task=classify_v2_for_eval,
    scores=[ExactMatch],
    metadata={
        "task_model": "gpt-4o-mini",
        "prompt_version": "v2-focused-disambiguation",
        "dataset": REGRESSION_DATASET_NAME,
        "purpose": "regression-test",
    },
)

print(regression_eval)

Experiment classifier-v2-regression-suite is running at https://www.braintrust.dev/app/The%20Gen%20Academy/p/customer-support-braintrust-evals/experiments/classifier-v2-regression-suite
customer-support-braintrust-evals [experiment_name=classifier-v2-regression-suite] (data): 8it [00:00, 27822.91it/s]


customer-support-braintrust-evals [experiment_name=classifier-v2-regression-suite] (tasks):   0%|          | 0…


=========================SUMMARY=========================
classifier-v2-regression-suite compared to classifier-v2-focused-prompt:
100.00% (-) 'ExactMatch' score	(0 improvements, 0 regressions)

1 (-) 'llm_calls'                      	(0 improvements, 0 regressions)
0 (-) 'tool_calls'                     	(0 improvements, 0 regressions)
0 (-) 'errors'                         	(0 improvements, 0 regressions)
0 (-) 'llm_errors'                     	(0 improvements, 0 regressions)
0 (-) 'tool_errors'                    	(0 improvements, 0 regressions)
307.50tok (-) 'prompt_tokens'                  	(0 improvements, 0 regressions)
0tok (-) 'prompt_cached_tokens'           	(0 improvements, 0 regressions)
0tok (-) 'prompt_cache_creation_tokens'   	(0 improvements, 0 regressions)
0tok (-) 'prompt_cache_creation_5m_tokens'	(0 improvements, 0 regressions)
0tok (-) 'prompt_cache_creation_1h_tokens'	(0 improvements, 0 regressions)
25.38tok (-12.50%) 'completion_tokens'              	(1 improvem

## 17. Define LLM judges

Now, we introduce LLM-as-a-judge.

The judge is not treated as ground truth. We will compare it against the human-reviewed subset.

Two judge models are included so we can also inspect judge-model differences.

In [20]:
import os
import openai
from autoevals import LLMClassifier

JUDGE_PROMPT = """You are evaluating an e-commerce support ticket classifier.

Allowed categories:
- order_status
- refund_request
- product_issue
- account_help
- other

Customer ticket:
{{input}}

Classifier prediction:
{{output}}

Human-reviewed expected category:
{{expected}}

Decide whether the classifier prediction is correct based on the customer's PRIMARY intent.

Important:
- Output 1 if the classifier prediction is correct.
- Output 0 if the classifier prediction is incorrect.
- A missing/undelivered order is primarily order_status even if a refund is requested.
- A damaged/wrong/defective item is primarily product_issue even if a refund is requested.

Answer using the required choice only.
"""

openai_client = openai.AsyncOpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    base_url="https://api.openai.com/v1",
)

judge_gpt4o = LLMClassifier(
    name="ClassifierJudge-gpt-4o",
    prompt_template=JUDGE_PROMPT,
    choice_scores={"1": 1, "0": 0},
    model="gpt-4o",
    use_cot=True,
    client=openai_client,
)

judge_gpt4o_mini = LLMClassifier(
    name="ClassifierJudge-gpt-4o-mini",
    prompt_template=JUDGE_PROMPT,
    choice_scores={"1": 1, "0": 0},
    model="gpt-4o-mini",
    use_cot=True,
    client=openai_client,
)

print("LLM judges ready.")

LLM judges ready.


## 18. Judge calibration uses the same human-reviewed cases

We now reuse the **same dataset that was reviewed by a human**.

This is important: the human reviewer and the LLM judge are evaluating the same set of examples, so their decisions can be compared without creating a second calibration sample.

The human feedback remains in Braintrust's Review workflow. The automated judge scores are produced below as separate experiments against the same review dataset.

## 19. LLM-as-a-judge calibration

Run each judge on the human-reviewed dataset.

The purpose is not to improve the classifier here. We are evaluating the **evaluator**.

After these runs, Braintrust contains:

```text
Same review cases
      │
      ├── HumanCorrectness      (entered in Human Review)
      ├── GPT-4o judge score
      └── GPT-4o-mini judge score
```

Use the Braintrust UI to inspect rows where the human score and automated judge disagree.

In [26]:
import json
import pandas as pd

human_review = pd.read_csv("data/human-review-v1.csv")

# Extract the actual human score entered in Braintrust
human_review["human_correctness"] = human_review["scores"].apply(
    lambda x: json.loads(x).get("HumanCorrectness")
)

# Ticket ID is stored in metadata
human_review["ticket_id"] = human_review["metadata"].apply(
    lambda x: json.loads(x).get("ticket_id")
)

display(
    human_review[
        [
            "ticket_id",
            "input",
            "output",
            "HumanExpectedCategory",
            "human_correctness",
            "ReviewerNotes",
        ]
    ]
)

,ticket_id,input,output,HumanExpectedCategory,human_correctness,ReviewerNotes
0,t004,"""I never received my order and I want my money...","""refund_request""","{""HumanExpectedCategory"":""order_status""}",0,It is a type of ambiguous case where the keywo...
1,t005,"""I didn't get my order, and I'm requesting a r...","""refund_request""","{""HumanExpectedCategory"":""order_status""}",0,The customer was more concerned with the undel...
2,t006,"""My order hasn't arrived, so I'd like to initi...","""refund_request""","{""HumanExpectedCategory"":""order_status""}",0,Concern was more for order not arriving.
3,t007,"""I haven't received my package and I would lik...","""refund_request""","{""HumanExpectedCategory"":""order_status""}",0,Concern was regarding receiving package more.
4,t044,"""My laptop arrived damaged and I want a full r...","""refund_request""","{""HumanExpectedCategory"":""product_issue""}",0,Laptop was damaged and need to be registered.
5,t045,"""I received a damaged laptop and would like to...","""refund_request""","{""HumanExpectedCategory"":""product_issue""}",0,NaN
6,t046,"""The laptop I ordered arrived in poor conditio...","""refund_request""","{""HumanExpectedCategory"":""product_issue""}",0,NaN
7,t047,"""Unfortunately, my laptop came damaged, and I ...","""refund_request""","{""HumanExpectedCategory"":""product_issue""}",0,NaN
8,t048,"""You sent me a size medium shirt but I ordered...","""product_issue""",product_issue,1,NaN
9,t026,"""Could you please check on my refund? I return...","""refund_request""",refund_request,1,NaN


In [23]:
judge_eval_gpt4o = await EvalAsync(
    PROJECT_NAME,
    experiment_name="judge-calibration-gpt-4o",
    data=human_review_dataset,
    task=classify_review_case,
    scores=[judge_gpt4o],
    metadata={
        "task_model": "gpt-4o-mini",
        "judge_model": "gpt-4o",
        "purpose": "judge-calibration",
        "reference": "braintrust-human-review",
    },
)

judge_eval_mini = await EvalAsync(
    PROJECT_NAME,
    experiment_name="judge-calibration-gpt-4o-mini",
    data=human_review_dataset,
    task=classify_review_case,
    scores=[judge_gpt4o_mini],
    metadata={
        "task_model": "gpt-4o-mini",
        "judge_model": "gpt-4o-mini",
        "purpose": "judge-calibration",
        "reference": "braintrust-human-review",
    },
)

print("Judge calibration experiments submitted.")

Experiment judge-calibration-gpt-4o-44a73306 is running at https://www.braintrust.dev/app/The%20Gen%20Academy/p/customer-support-braintrust-evals/experiments/judge-calibration-gpt-4o-44a73306
customer-support-braintrust-evals [experiment_name=judge-calibration-gpt-4o] (data): 20it [00:00, 28111.96it/s]


customer-support-braintrust-evals [experiment_name=judge-calibration-gpt-4o] (tasks):   0%|          | 0/20 [0…


=========================SUMMARY=========================
judge-calibration-gpt-4o-44a73306 compared to judge-calibration-gpt-4o-mini:
60.00% 'ClassifierJudge-gpt-4o' score

2 (-) 'llm_calls'                            	(0 improvements, 0 regressions)
0 (-) 'tool_calls'                           	(0 improvements, 0 regressions)
0 (-) 'errors'                               	(0 improvements, 0 regressions)
0 (-) 'llm_errors'                           	(0 improvements, 0 regressions)
0 (-) 'tool_errors'                          	(0 improvements, 0 regressions)
212.20tok (-) 'prompt_tokens'                        	(0 improvements, 0 regressions)
0tok (-) 'prompt_cached_tokens'                 	(0 improvements, 0 regressions)
0tok (-) 'prompt_cache_creation_tokens'         	(0 improvements, 0 regressions)
0tok (-) 'prompt_cache_creation_5m_tokens'      	(0 improvements, 0 regressions)
0tok (-) 'prompt_cache_creation_1h_tokens'      	(0 improvements, 0 regressions)
23.25tok (+05.00%) 'compl

Experiment judge-calibration-gpt-4o-mini-a0c03441 is running at https://www.braintrust.dev/app/The%20Gen%20Academy/p/customer-support-braintrust-evals/experiments/judge-calibration-gpt-4o-mini-a0c03441
customer-support-braintrust-evals [experiment_name=judge-calibration-gpt-4o-mini] (data): 20it [00:00, 54330.36it/s]


customer-support-braintrust-evals [experiment_name=judge-calibration-gpt-4o-mini] (tasks):   0%|          | 0/…


=========================SUMMARY=========================
judge-calibration-gpt-4o-mini-a0c03441 compared to judge-calibration-gpt-4o-44a73306:
60.00% 'ClassifierJudge-gpt-4o-mini' score

2 (-) 'llm_calls'                            	(0 improvements, 0 regressions)
0 (-) 'tool_calls'                           	(0 improvements, 0 regressions)
0 (-) 'errors'                               	(0 improvements, 0 regressions)
0 (-) 'llm_errors'                           	(0 improvements, 0 regressions)
0 (-) 'tool_errors'                          	(0 improvements, 0 regressions)
212.20tok (-) 'prompt_tokens'                        	(0 improvements, 0 regressions)
0tok (-) 'prompt_cached_tokens'                 	(0 improvements, 0 regressions)
0tok (-) 'prompt_cache_creation_tokens'         	(0 improvements, 0 regressions)
0tok (-) 'prompt_cache_creation_5m_tokens'      	(0 improvements, 0 regressions)
0tok (-) 'prompt_cache_creation_1h_tokens'      	(0 improvements, 0 regressions)
23.35tok (+

In [27]:
def judge_agreement(eval_result, scorer_name):
    judge_rows = []

    for r in eval_result.results:
        judge_rows.append({
            "ticket_id": r.metadata["ticket_id"],
            "judge_score": r.scores.get(scorer_name),
        })

    judge_df = pd.DataFrame(judge_rows)

    comparison = human_review[
        ["ticket_id", "human_correctness"]
    ].merge(
        judge_df,
        on="ticket_id"
    )

    comparison["agreement"] = (
        comparison["human_correctness"]
        == comparison["judge_score"]
    )

    print(
        f"{scorer_name} human agreement: "
        f"{comparison['agreement'].mean():.2%}"
    )

    return comparison

In [28]:
gpt4o_comparison = judge_agreement(
    judge_eval_gpt4o,
    "ClassifierJudge-gpt-4o"
)

mini_comparison = judge_agreement(
    judge_eval_mini,
    "ClassifierJudge-gpt-4o-mini"
)

display(gpt4o_comparison)
display(mini_comparison)

ClassifierJudge-gpt-4o human agreement: 100.00%
ClassifierJudge-gpt-4o-mini human agreement: 100.00%


,ticket_id,human_correctness,judge_score,agreement
0,t004,0,0,True
1,t005,0,0,True
2,t006,0,0,True
3,t007,0,0,True
4,t044,0,0,True
5,t045,0,0,True
6,t046,0,0,True
7,t047,0,0,True
8,t048,1,1,True
9,t026,1,1,True


,ticket_id,human_correctness,judge_score,agreement
0,t004,0,0,True
1,t005,0,0,True
2,t006,0,0,True
3,t007,0,0,True
4,t044,0,0,True
5,t045,0,0,True
6,t046,0,0,True
7,t047,0,0,True
8,t048,1,1,True
9,t026,1,1,True


## 20. Compare human and automated evaluation in Braintrust

The calibration comparison is intentionally done in the Braintrust UI because that is where the native human-review scores live.

### What to inspect

For the same reviewed examples, compare:

- **HumanCorrectness**
- **ClassifierJudge-gpt-4o**
- **ClassifierJudge-gpt-4o-mini**

Focus on:

- agreement between human and judge
- cases where the human says `Incorrect` but the judge says `Correct`
- cases where the human says `Correct` but the judge says `Incorrect`
- scorer errors or missing judge choices
- recurring ambiguity in the ticket taxonomy

Braintrust keeps human and automated scoring in the same project, as demonstrated. The goal is to diagnose disagreement and refine the judge rubric only if a clear pattern appears.

## 21. Evaluation lifecycle

The final workflow now follows the hybrid evaluation approach:

```text
Golden dataset
      ↓
Deterministic baseline
      ↓
Failure analysis
      ↓
Human-review sample
      ↓
Trusted human feedback
      ↓
Classifier iteration + regression
      ↓
LLM judge on reviewed cases
      ↓
Human ↔ judge comparison
      ↓
Refine automated evaluation if needed
```

Humans are not required to annotate every run. Their role is to establish and periodically refresh the evaluation standard on representative or difficult examples.

## 22. Evaluation lifecycle demonstrated

| Stage | What this notebook does |
|---|---|
| Golden dataset | Create 100 labeled customer-support tickets |
| Baseline metric | ExactMatch only |
| Baseline | Run V1 against the golden dataset |
| Failure analysis | Confusion matrix + failed examples |
| Human review | Review failures + sampled correct cases in Braintrust UI |
| Iterate | Add primary-intent rules in V2 |
| Compare | V1 vs V2 score changes, wins, and regressions |
| Regression | Promote known failures into a reusable regression dataset |
| LLM-as-a-judge | Run judges separately from the classifier baseline |
| Calibration | Compare judge scores with native Braintrust human feedback |

This keeps **model evaluation** separate from **evaluator calibration**.

## 23. Braintrust features deliberately tested

| Feature | Why we used it |
|---|---|
| Tracing | Inspect the existing LangGraph execution |
| Golden dataset | Reuse the same benchmark across experiments |
| ExactMatch | Deterministic classification baseline |
| Experiments | Compare V1 and V2 |
| Human Review | Capture trusted human feedback directly in Braintrust |
| Review dataset | Reuse the same reviewed cases for calibration |
| LLMClassifier | Automated LLM-as-a-judge evaluation |
| Judge calibration | Compare automated judgment against human judgment |
| Regression dataset | Convert known failures into reusable guardrails |